# DGCA Fusion на DUSHA: BERT (text) + WavLM (audio)

Адаптация **Dimension-Wise Gated Cross-Attention** (DGCA, WWW'25) для задачи SER.  
Оригинал: BERT + ResNet50 (текст + изображение).  
Здесь: BERT + WavLM (текст + аудио) на датасете DUSHA (5 классов).

```
h_text  = BERT(X)[CLS]          ∈ R^768
h_audio = MeanPool(WavLM(wav))  ∈ R^768

T = W_t · h_text  + b_t    ∈ R^D       (проекция)
A = W_a · h_audio + b_a    ∈ R^D

T_ref = LayerNorm(T + MHA(T, A, A))    (cross-attention)
A_ref = LayerNorm(A + MHA(A, T, T))

α_text(d)  = exp(G_t(d)) / (exp(G_t(d)) + exp(G_a(d)))   (gating)
α_audio(d) = 1 - α_text(d)

F(d) = α_text(d)·T_ref(d) + α_audio(d)·A_ref(d)          (fusion)
logits = W_c · F + b_c
```

**Входные .pt файлы:**
- `BERT_CKPT_DIR` — директория с сохранённой BERT моделью (save_pretrained)
- `WAVLM_CKPT`   — путь к WavLM чекпоинту (наш формат с config внутри)

## 1. Install & clone

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'peft>=0.10', 'soundfile', 'torchaudio',
    'librosa', 'scikit-learn', 'tqdm', 'pyyaml',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done. CWD:', os.getcwd())

## 2. Imports & config

In [ ]:
import warnings, pathlib, random
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, accuracy_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── чекпоинты предобученных моделей ──────────────────────────────────────────
BERT_CKPT_DIR = '/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints/best_bert_dusha'
WAVLM_CKPT    = '/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints/wavlm_dusha_majority.pt'

# ── пути к данным ─────────────────────────────────────────────────────────────
AGG_ROOT        = pathlib.Path('/kaggle/input/datasets/aleksandribryanov/agg-dusha')
AUDIO_TRAIN_DIR = pathlib.Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_train')
AUDIO_TEST_DIR  = pathlib.Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_test')
TRAIN_TSV       = AGG_ROOT / 'aggregated_ds_0.9.tsv'
TEST_TSV        = AGG_ROOT / 'aggregated_ds_0.9_test.tsv'
OUT_DIR         = pathlib.Path('/kaggle/working')

# ── гиперпараметры ────────────────────────────────────────────────────────────
TRAIN_FRACTION  = 0.05
VAL_FRACTION    = 0.15
FUSION_DIM      = 1024
NUM_HEADS       = 4
BATCH_SIZE      = 16
FUSION_DROP     = 0.3
LR_FUSION       = 1e-4
LR_BACKBONE     = 2e-5
WEIGHT_DECAY    = 1e-2
WARMUP_STEPS    = 100
MAX_STEPS       = 10_000
EVAL_EVERY      = 500
ES_PATIENCE     = 6
MAX_TEXT_LEN    = 128
MAX_AUDIO_S     = 10.0
SR_TARGET       = 16_000
SEED            = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']
NUM_CLASSES    = len(DUSHA_LABELS)

## 3. Загрузка данных

In [ ]:
def load_tsv(tsv_path, audio_dir, fraction=None):
    df = pd.read_csv(tsv_path, sep='\t')
    df = df[df['aggregated_emo'].isin(DUSHA_LABEL2ID)]
    df = df[df['speaker_text'].notna() & (df['speaker_text'].str.strip() != '')]
    if fraction:
        df = df.sample(frac=fraction, random_state=SEED)
    records, missing = [], 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=tsv_path.name, leave=False):
        path = audio_dir / row['audio_path']
        if path.exists():
            records.append({
                'text':  row['speaker_text'],
                'path':  str(path),
                'label': DUSHA_LABEL2ID[row['aggregated_emo']],
            })
        else:
            missing += 1
    print(f'  {tsv_path.name}: {len(records)} records ({missing} missing)')
    return records


all_train = load_tsv(TRAIN_TSV, AUDIO_TRAIN_DIR, fraction=TRAIN_FRACTION)
test_recs = load_tsv(TEST_TSV,  AUDIO_TEST_DIR)

train_recs, val_recs = train_test_split(
    all_train, test_size=VAL_FRACTION, random_state=SEED,
    stratify=[r['label'] for r in all_train],
)
print(f'\nTrain: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}')

## 4. Загрузка предобученных моделей

In [ ]:
from src.models import build_model
from src.config import ExperimentConfig

# ── BERT backbone (.safetensors через save_pretrained директорию) ─────────────
print(f'Loading BERT from {BERT_CKPT_DIR}')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CKPT_DIR)
bert_backbone  = AutoModel.from_pretrained(BERT_CKPT_DIR).to(device)
BERT_DIM = bert_backbone.config.hidden_size
print(f'  BERT hidden_size: {BERT_DIM}')

# ── WavLM backbone (.pt файл нашего формата) ─────────────────────────────────
print(f'Loading WavLM from {WAVLM_CKPT}')
ckpt = torch.load(WAVLM_CKPT, map_location=device, weights_only=False)
wavlm_cfg = ExperimentConfig()
for k, v in ckpt.get('config', {}).items():
    if hasattr(wavlm_cfg, k): setattr(wavlm_cfg, k, v)
wavlm_full = build_model(wavlm_cfg)
wavlm_full.load_state_dict(ckpt['model_state_dict'])
wavlm_full.to(device)
wavlm_backbone = wavlm_full.backbone
WAVLM_DIM = wavlm_backbone.config.hidden_size
print(f'  WavLM hidden_size: {WAVLM_DIM}')

# ── Заморозить оба backbone ───────────────────────────────────────────────────
for p in bert_backbone.parameters():  p.requires_grad = False
for p in wavlm_backbone.parameters(): p.requires_grad = False
bert_backbone.eval()
wavlm_backbone.eval()
print('Backbones frozen — only DGCA fusion will be trained.')

## 5. DGCA Fusion модуль

In [ ]:
class DGCAFusion(nn.Module):
    """
    Dimension-Wise Gated Cross-Attention fusion (DGCA, WWW'25).
    Адаптация для Text (BERT) + Audio (WavLM).

    Шаги:
      1. Projection:   T = W_t·h_text + b_t,  A = W_a·h_audio + b_a   → R^D
      2. Cross-Attn:   T_ref = LN(T + MHA(T,A,A)), A_ref = LN(A + MHA(A,T,T))
      3. Gating:       α_t(d) = softmax([G_t(d), G_a(d)])[0] per dimension d
      4. Fusion:       F(d) = α_t(d)·T_ref(d) + α_a(d)·A_ref(d)
      5. Classify:     logits = W_c·F + b_c
    """
    def __init__(self, d_text, d_audio, D=512, num_heads=4, num_classes=5, dropout=FUSION_DROP):
        super().__init__()
        # 1. Projection
        self.proj_text  = nn.Linear(d_text,  D)
        self.proj_audio = nn.Linear(d_audio, D)

        # 2. Cross-Attention (bidirectional)
        self.mha_t2a = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.mha_a2t = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.ln_text  = nn.LayerNorm(D)
        self.ln_audio = nn.LayerNorm(D)

        # 3. Dimension-wise gating
        self.gate_text  = nn.Linear(D, D)
        self.gate_audio = nn.Linear(D, D)

        # 5. Classifier
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(D, num_classes)

    def forward(self, h_text, h_audio):
        """
        h_text  : (B, d_text)
        h_audio : (B, d_audio)
        returns : logits (B, num_classes)
        """
        # 1. Projection → (B, 1, D)
        T = self.proj_text(h_text).unsqueeze(1)
        A = self.proj_audio(h_audio).unsqueeze(1)

        # 2. Bidirectional cross-attention
        T_ca, _ = self.mha_t2a(T, A, A)          # text queries audio
        T_ref   = self.ln_text(T + T_ca)          # residual + LN
        A_ca, _ = self.mha_a2t(A, T, T)          # audio queries text
        A_ref   = self.ln_audio(A + A_ca)

        T_ref = T_ref.squeeze(1)                  # (B, D)
        A_ref = A_ref.squeeze(1)

        # 3. Dimension-wise gating logits
        G_t = self.gate_text(T_ref)               # (B, D)
        G_a = self.gate_audio(A_ref)              # (B, D)

        # softmax across modalities per dimension
        gates   = F.softmax(torch.stack([G_t, G_a], dim=-1), dim=-1)  # (B, D, 2)
        alpha_t = gates[..., 0]                   # (B, D)
        alpha_a = gates[..., 1]

        # 4. Dimension-wise fusion
        fused = alpha_t * T_ref + alpha_a * A_ref  # (B, D)

        # 5. Classification
        return self.classifier(self.dropout(fused))


fusion = DGCAFusion(
    d_text=BERT_DIM, d_audio=WAVLM_DIM,
    D=FUSION_DIM, num_heads=NUM_HEADS, num_classes=NUM_CLASSES,
).to(device)

total = sum(p.numel() for p in fusion.parameters())
print(f'DGCA fusion params: {total:,}')

## 6. Dataset & DataLoader

In [ ]:
MAX_AUDIO_LEN = int(MAX_AUDIO_S * SR_TARGET)


class MultimodalDataset(Dataset):
    def __init__(self, records, tokenizer, max_text_len=MAX_TEXT_LEN):
        self.records  = records
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len

    def __len__(self): return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        enc = self.tokenizer(
            r['text'], truncation=True, padding='max_length',
            max_length=self.max_text_len, return_tensors='pt'
        )
        wav, sr = sf.read(r['path'], dtype='float32')
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        if len(wav) > MAX_AUDIO_LEN:
            wav = wav[:MAX_AUDIO_LEN]
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'audio':          torch.tensor(wav, dtype=torch.float32),
            'label':          torch.tensor(r['label'], dtype=torch.long),
        }


def collate_fn(batch):
    """Pad audio sequences to max length in batch."""
    max_len = max(b['audio'].shape[0] for b in batch)
    audios  = torch.zeros(len(batch), max_len)
    masks   = torch.zeros(len(batch), max_len)
    for i, b in enumerate(batch):
        L = b['audio'].shape[0]
        audios[i, :L] = b['audio']
        masks[i,  :L] = 1.0
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'audio':          audios,
        'audio_mask':     masks,
        'label':          torch.stack([b['label']          for b in batch]),
    }


train_ds = MultimodalDataset(train_recs, bert_tokenizer)
val_ds   = MultimodalDataset(val_recs,   bert_tokenizer)
test_ds  = MultimodalDataset(test_recs,  bert_tokenizer)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f'Train batches: {len(train_ld)}  Val: {len(val_ld)}  Test: {len(test_ld)}')

## 7. Encoder helpers

In [ ]:
from transformers import AutoFeatureExtractor

wavlm_processor = AutoFeatureExtractor.from_pretrained(
    wavlm_cfg.processor_name or wavlm_cfg.model_name)


@torch.no_grad()
def encode_text(input_ids, attention_mask):
    """BERT → CLS embedding: (B, BERT_DIM)"""
    out = bert_backbone(input_ids=input_ids, attention_mask=attention_mask)
    return out.last_hidden_state[:, 0, :]


@torch.no_grad()
def encode_audio(audio, audio_mask):
    """WavLM → mean-pooled embedding: (B, WAVLM_DIM)"""
    embeddings = []
    for i in range(audio.shape[0]):
        wav_np = audio[i][audio_mask[i].bool()].cpu().numpy()
        inputs = wavlm_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
        hidden = wavlm_backbone(inputs['input_values'].to(device)).last_hidden_state
        embeddings.append(hidden.mean(dim=1).squeeze(0))
    return torch.stack(embeddings)

## 8. Обучение

In [ ]:
optimizer = torch.optim.AdamW(
    fusion.parameters(), lr=LR_FUSION, weight_decay=WEIGHT_DECAY)
scheduler_warmup = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS)
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.3, patience=1)

criterion = nn.CrossEntropyLoss()

best_val_wacc = -1.0
es_counter    = 0
global_step   = 0
history       = []
train_iter    = iter(train_ld)


@torch.no_grad()
def evaluate(loader):
    fusion.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        audio = batch['audio'].to(device)
        amask = batch['audio_mask'].to(device)
        labs  = batch['label'].to(device)
        logits = fusion(encode_text(ids, mask), encode_audio(audio, amask))
        total_loss  += criterion(logits, labs).item() * len(labs)
        preds_all.append(logits.argmax(-1).cpu().numpy())
        labels_all.append(labs.cpu().numpy())
    val_loss = total_loss / len(loader.dataset)
    val_wacc = balanced_accuracy_score(
        np.concatenate(labels_all), np.concatenate(preds_all))
    fusion.train()
    return val_loss, val_wacc


print(f'Backbones frozen. Training only DGCA fusion ({sum(p.numel() for p in fusion.parameters()):,} params)')
print(f'max_steps={MAX_STEPS}  eval_every={EVAL_EVERY}  es={ES_PATIENCE}  save_by=val_wacc')
print('-' * 70)

fusion.train()
running_loss = 0.0

while global_step < MAX_STEPS:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_ld)
        batch = next(train_iter)

    ids   = batch['input_ids'].to(device)
    mask  = batch['attention_mask'].to(device)
    audio = batch['audio'].to(device)
    amask = batch['audio_mask'].to(device)
    labs  = batch['label'].to(device)

    optimizer.zero_grad()
    logits = fusion(encode_text(ids, mask), encode_audio(audio, amask))
    loss   = criterion(logits, labs)
    loss.backward()
    nn.utils.clip_grad_norm_(fusion.parameters(), 1.0)
    optimizer.step()
    scheduler_warmup.step()
    running_loss += loss.item()
    global_step  += 1

    if global_step % EVAL_EVERY == 0:
        avg_loss = running_loss / EVAL_EVERY
        val_loss, val_wacc = evaluate(val_ld)
        prev_lr  = optimizer.param_groups[0]['lr']
        scheduler_plateau.step(val_wacc)
        cur_lr   = optimizer.param_groups[0]['lr']
        running_loss = 0.0

        is_best = val_wacc > best_val_wacc
        if is_best:
            best_val_wacc = val_wacc
            es_counter    = 0
            torch.save({
                'fusion':   fusion.state_dict(),
                'step':     global_step,
                'val_wacc': val_wacc,
            }, str(OUT_DIR / 'best_dgca_fusion.pt'))
        else:
            es_counter += 1

        lr_info = f'{cur_lr:.2e}' + (' ↓' if cur_lr < prev_lr else '')
        history.append((global_step, avg_loss, val_loss, val_wacc))
        print(
            f'Step {global_step:6d}  train={avg_loss:.4f}  '
            f'val_loss={val_loss:.4f}  val_wacc={val_wacc:.4f}  '
            f'lr={lr_info}' + ('  *' if is_best else ''),
            flush=True,
        )

        if es_counter >= ES_PATIENCE:
            print(f'Early stopping at step {global_step}')
            break

print(f'\nBest val_wacc: {best_val_wacc:.4f}')
print(f'Saved → {OUT_DIR / "best_dgca_fusion.pt"}')

## 9. Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

steps      = [h[0] for h in history]
tr_losses  = [h[1] for h in history]
val_losses = [h[2] for h in history]
val_waccs  = [h[3] for h in history]
best_step  = steps[int(np.argmin(val_losses))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(steps, tr_losses,  label='train loss', color='steelblue')
ax1.plot(steps, val_losses, label='val loss',   color='darkorange')
ax1.axvline(best_step, color='red', linestyle='--', alpha=0.6, label=f'best={best_step}')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(steps, val_waccs, color='green', label='val wacc')
ax2.axvline(best_step, color='red', linestyle='--', alpha=0.6)
ax2.set_xlabel('Step'); ax2.set_ylabel('Weighted Accuracy')
ax2.set_title('Val Weighted Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'dgca_curves.png'), dpi=150)
plt.show()

## 10. Финальная оценка на test

In [ ]:
ckpt_best = torch.load(str(OUT_DIR / 'best_dgca_fusion.pt'), map_location=device)
fusion.load_state_dict(ckpt_best['fusion'])
print(f'Loaded best checkpoint from step {ckpt_best["step"]}  val_wacc={ckpt_best["val_wacc"]:.4f}')

fusion.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for batch in tqdm(test_ld, desc='Test'):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        audio = batch['audio'].to(device)
        amask = batch['audio_mask'].to(device)
        preds_all.append(fusion(encode_text(ids, mask), encode_audio(audio, amask)).argmax(-1).cpu().numpy())
        labels_all.append(batch['label'].numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)

print('\n=== Test Results ===')
print(f'Accuracy          : {accuracy_score(labels, preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(labels, preds):.4f}')
print()
print(classification_report(labels, preds, target_names=DUSHA_LABELS, zero_division=0))